## KNN Classification — Titanic Survival Prediction

This notebook continues from Project 01, where I explored the Titanic dataset
through EDA. The goal there was to understand the data — distributions, missing
values, and which features correlated with survival. This notebook uses those
same insights to build a predictive model.

The algorithm is **K-Nearest Neighbors (KNN)**: to classify a new passenger,
the model finds the K most similar passengers in the training set and takes a
majority vote. No equation is learned — the training data itself is the model.

This also marks the transition from **regression** (predicting a continuous
number, like house price) to **classification** (predicting a category —
survived or not).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics

In [ ]:
df = pd.read_csv('../data/raw/train.csv')
print(df.shape)  
print(df.isnull().sum())

### Feature Selection and Preprocessing

Five features chosen based on the Project 01 EDA heatmap and survival plots:
`Pclass`, `Sex`, `Age`, `SibSp`, and `Fare`. The `survival_by_pclass.png` and
`survival_by_sex.png` visuals already showed these as the strongest signals.

Two preprocessing steps are required before modeling:
- `Sex` is categorical text — mapped to 0 (male) and 1 (female) so the model
  can compute distances with it
- `Age` has 177 null values — filled with the median rather than dropping rows,
  which would lose too much data

The `y.value_counts()` output establishes the **baseline accuracy**: 549 of 891
passengers did not survive (~61.5%). A model that always predicts "did not
survive" would reach 61.5% accuracy without learning anything. Every model
built here must beat that number to be meaningful.

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Fare']
df_model = df[features + ['Survived']].copy()
df_model['Sex'] = df_model['Sex'].map({'male': 0, 'female': 1})
df_model['Age'] = df_model['Age'].fillna(df_model['Age'].median())
df_model.dropna(inplace=True)

x = df_model[features]
y = df_model['Survived']

print(x.shape)   
print(y.shape) 
print(y.value_counts()) 

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x_train.shape)   
print(x_test.shape)    

### Feature Scaling

KNN classifies by distance. If features are on different scales, the feature
with the largest numerical range will dominate the distance calculation
regardless of its actual importance.

In this dataset: `Fare` ranges from 0 to 512, while `Sex` is only 0 or 1.
Without scaling, two passengers with identical profiles but different fares
would appear far apart, while two passengers with completely different profiles
but similar fares would appear close. KNN would effectively become
"K-Nearest Fares."

`StandardScaler` converts each feature to its z-score: `(x − mean) / std`.
After scaling every feature has mean 0 and standard deviation 1, so all five
features contribute equally to the distance calculation.

**Critical distinction:**
- `fit_transform(X_train)` — learns the mean and std from training data, then
  applies the formula to X_train
- `transform(X_test)` — applies the *same* training statistics to X_test,
  without re-learning anything from it

Re-fitting the scaler on test data would be **data leakage**: the model's
evaluation pipeline would have seen test data before prediction, making
accuracy look better than it would on truly unseen data.

In [ ]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)   
x_test_scaled  = scaler.transform(x_test)        

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train_scaled, y_train)

### Predict and Evaluate

Predictions are made on `X_test_scaled` — the test data must go through the
same scaling that was applied to training data. Passing raw `X_test` would
mean the model is measuring distances in unscaled space after being trained
on scaled space.

**Accuracy** measures the fraction of correct predictions overall. The
**confusion matrix** breaks this down further:

|  | Predicted: No | Predicted: Yes |
|---|---|---|
| **Actual: No** | True Negative | False Positive (Type I error) |
| **Actual: Yes** | False Negative (Type II error) | True Positive |

The diagonal cells are correct predictions. Off-diagonal cells are errors —
and the two types of error have different real-world costs depending on context.

In [ ]:
y_pred = knn.predict(x_test_scaled) 

accuracy = metrics.accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
cm = metrics.confusion_matrix(y_test, y_pred)
print(cm)
print(metrics.classification_report(y_test, y_pred, target_names=['Did not survive', 'Survived']))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Did not survive', 'Survived'],
            yticklabels=['Did not survive', 'Survived'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('KNN Confusion Matrix (K=5)')
plt.tight_layout()
plt.savefig('../visuals/knn_confusion_matrix.png', dpi=150)
plt.show()

### Improvement — Finding the Optimal K

K=5 was a reasonable guess, not a reasoned choice. To find the best K, we
loop through K values from 1 to 25 and record both training and test accuracy
at each step.

The two-line plot reveals the **bias-variance tradeoff** directly:
- At K=1, training accuracy is 1.0 — the nearest neighbor of any training
  point is itself, so the model memorizes perfectly. Test accuracy is much
  lower: overfitting.
- As K grows, training accuracy drops and test accuracy rises — the model
  becomes less sensitive to individual points.
- At very large K, both lines flatten or decline — the model stops
  distinguishing between inputs and underfits.

The best K is where test accuracy peaks.

In [ ]:
k_range = range(1, 26)
train_scores = []
test_scores  = []

for k in k_range:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(x_train_scaled, y_train)
    train_scores.append(knn_k.score(x_train_scaled, y_train))
    test_scores.append(knn_k.score(x_test_scaled, y_test))

plt.figure(figsize=(10, 5))
plt.plot(k_range, train_scores, label='Training accuracy', color='steelblue')
plt.plot(k_range, test_scores,  label='Test accuracy',     color='seagreen')
plt.xlabel('K (number of neighbors)')
plt.ylabel('Accuracy')
plt.title('KNN — Accuracy vs K')
plt.legend()
plt.xticks(k_range)
plt.tight_layout()
plt.savefig('../visuals/knn_accuracy_vs_k.png', dpi=150)
plt.show()

In [ ]:
best_k = test_scores.index(max(test_scores)) + 1
print(f"Best K: {best_k}")

knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(x_train_scaled, y_train)
y_pred_best = knn_best.predict(x_test_scaled)
print(f"Best K accuracy: {metrics.accuracy_score(y_test, y_pred_best):.4f}")

## Conclusion

### Data and baseline
The training set contained 891 passengers: 549 did not survive (61.6%) and
342 survived (38.4%). This 62/38 split sets the baseline — any model must
exceed 61.6% accuracy to be better than always predicting the majority class.

### K=5 baseline model
The initial model achieved **79.3% accuracy** — roughly 18 percentage points
above the naive baseline. The confusion matrix (K=5) broke down as:

- **92 True Negatives** — correctly predicted did not survive
- **50 True Positives** — correctly predicted survived
- **13 False Positives** — predicted survived, actually did not (Type I error)
- **24 False Negatives** — predicted did not survive, actually did (Type II error)

The model's **recall was asymmetric**: 88% for "did not survive" vs 68% for
"survived." It was meaningfully better at identifying deaths than survivors.
This is expected — 62% of training examples are deaths, so the model has seen
far more examples of that class and has a stronger pattern to match.

The FN count (24) exceeding FP count (13) reflects this same asymmetry: the
model under-predicts survival more than it over-predicts it.

### Accuracy vs K plot
The plot confirmed the theoretical prediction exactly. Training accuracy
started near 0.975 at K=1 (the model memorizing its own training data) and
steadily declined. Test accuracy started low at K=1, rose as K increased, and
peaked in the K=7–9 range before flattening. The gap between the two lines
at small K is the overfitting signal — the model learned the training data's
noise rather than its signal.

### Optimised model (K=7)
Retraining with K=7 raised accuracy to **81.6%**, a 2.3 percentage point
improvement over K=5 with no new data and no new features — just a better
hyperparameter choice.

### Limitations
- Only 5 features used. `Embarked` and engineered features (e.g. family size
  from `SibSp + Parch`) could add signal.
- KNN assumes all features contribute equally after scaling. It cannot learn
  that `Sex` is a stronger predictor than `SibSp` — it only knows distance.
- KNN has no interpretability: there are no coefficients to read. We cannot
  say "being female increased survival odds by X." The next algorithms will
  begin to restore that.